In [ ]:
import numpy as np
from scipy import ndimage
import glob
import os
import pandas as pd



def analyze(file):
    # extract the label and index from the filename, and read the image data from the csv file
    filename = os.path.basename(file)
    parts = filename.split("_")
    label =  parts[1]
    index =  parts[2].split(".")[0]

    image = np.loadtxt(file, delimiter=',')

    nr_pix = np.sum(image == 0)

    rows_with_1 = 0
    for row in image:
        if(np.sum(row==0) == 1):
            rows_with_1+= 1
            
    cols_with_1=0
    for col in image.T:
        if(np.sum(col==0) == 1):
            cols_with_1+= 1
            
    rows_with_2=0
    for row in image:
        if(np.sum(row==0) == 2):
            rows_with_2+= 1

    cols_with_2=0
    for col in image.T:
        if(np.sum(col==0) == 2):
            cols_with_2+= 1

    rows_with_3p=0
    for row in image:
        if(np.sum(row==0) >= 3):
            rows_with_3p+= 1
            
    cols_with_3p=0
    for col in image.T:
        if(np.sum(col==0) >= 3):
            cols_with_3p+= 1

    rows_1p = np.where(image == 0)[0]
    height= max(rows_1p) - min(rows_1p)

    cols_1p = np.where(image == 0)[1]
    width= max(cols_1p) - min(cols_1p)

    aspect_ratio = width/height

    bp_sum_in_rows = np.sum(image == 0, axis=1)
    maxrow = max(bp_sum_in_rows)

    bp_sum_in_cols = np.sum(image == 0, axis=0)
    maxcol = max(bp_sum_in_cols)
    # labeled_black was not used 
    labeled_black, connected_areas = ndimage.label(image == 0)

    labeled_white, num_white = ndimage.label(image == 1, structure=[[0,1,0],[1,1,1],[0,1,0]])
    eyes = num_white-1

    hollowness= np.sum(labeled_white>1)/nr_pix

    #custom 
    img_bdensity = nr_pix/(height*width)

    return [label, index, nr_pix,
            rows_with_1, cols_with_1,
            rows_with_2, cols_with_2,
            rows_with_3p, cols_with_3p,
            height, width, aspect_ratio,
            maxrow, maxcol,
            connected_areas, eyes,
            hollowness, img_bdensity]
    
        



def main():
    
    src = "C:\\Users\\ammar\\Desktop\\A1_Ammar_Jileidan_40450780\\csvs/*.csv"

    # to save the results in a list of lists, which will be converted to a dataframe later
    results = []

    # loop through all the files in the source directory and analyze them, appending the results to the list
    for file in glob.glob(src):
        results.append(analyze(file))
    
    # create a dataframe from the results and save it to a csv file
    df = pd.DataFrame(results, columns = [
            "label", "index", "nr_pix",
            "rows_with_1", "cols_with_1",
            "rows_with_2", "cols_with_2",
            "rows_with_3p", "cols_with_3p",
            "height", "width", "aspect_ratio",
            "maxrow", "maxcol",
            "connected_areas", "eyes",
            "hollowness", "img_bdensity"])
    
    df = df.sort_values(by=["label", "index"])
    df.to_csv(os.path.join("C:\\Users\\ammar\\Desktop\\A1_Ammar_Jileidan_40450780", "40450780_features.csv"), index=False)

main()
